# 1. ZuCo paper data preparation — CPU only
Select **CPU** and **Run all**. This loads source EEG, maps sentences/fixations, filters, fits cleaning models and saves the accepted **individual raw and cleaned trials** to Drive. It runs **no permutations and no neural-network training**.

Wait for **CPU PREPARATION COMPLETE**. You may then disconnect/change runtimes and open the separate GPU notebook. Each completed CPU reader is saved independently; rerunning skips verified prepared readers. An interruption during an unfinished reader can still require repeating that reader's CPU work.

**Storage:** a **4 GB total cap** for this new workflow, including ~1 GB temporary permutation reserve. Saved CPU data is capped at approximately **2.7 GB**, losslessly compressed float32. No quantization or replacement by class averages. All 105 channels and 201 samples (−600…1000 ms, 125 Hz) are retained for each accepted raw/clean fixation trial. Labels, word/sentence/reader IDs, trial identities and pair indices are saved. These are the paper's accepted positive/negative trials; neutral trials are excluded by that paper policy.

**This is a new staged folder:** `Thesis/Results/word_sentiment_eeg/zuco_paper_figures_staged/`. The earlier reproduction outputs are preserved. Its incomplete coefficients alone cannot be promoted to this validated trial archive; the first staged CPU pass must build these files. Stop the old reproduction job before launching this workflow so both do not compete for storage. No files are deleted from the old folder.


## Connect Drive and load the pinned code
Choose the same account for Drive mount and quota authentication. No GitHub token is required.

In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/zuco_paper_figures_staged'
REPO=Path('/content/word-sentiment-eeg-paper-staged')
CODE_REVISION='9c057bd49d6e6398a59ca07629085315cdede794'
def git(*args):
    subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],check=True,
                   env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    git('-C',REPO,'checkout','--detach',CODE_REVISION)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-paper.txt')],check=True)
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# A mounted Drive filesystem disk_usage value can reflect the runtime volume;
# use the authenticated Drive quota instead.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();quota=response.json()['storageQuota']
print('Quota account:',response.json().get('user',{}).get('emailAddress','unavailable'))
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a finite storage quota; verify free space before starting.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
print(f'Account free space: {FREE_DRIVE_BYTES/1e9:.2f} GB. Workflow checks the additional space needed before source reads.')
print('Destination:',OUT)


## Prepare and save readers

In [ ]:
from datetime import datetime, timezone
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/zuco_paper_stages.py'),
         '--thesis-root',str(THESIS),'--phase','prepare',
         '--budget-bytes','4000000000','--free-drive-bytes',str(FREE_DRIVE_BYTES)]
attempt=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
run_with_progress(command,OUT/'status'/f'prepare_{attempt}.json',activity='ZuCo paper prepare: persistent CPU to GPU handoff')


## Verify that changing runtimes is safe

In [ ]:
state=json.loads((OUT/'cpu_preparation_manifest.json').read_text())
assert state['status']=='completed'
rows=[]
for subject in state['subjects']:
    item=json.loads((OUT/'readers'/subject/'prepared.json').read_text())
    rows.append(dict(reader=subject,negative=item['accepted_counts'][0],positive=item['accepted_counts'][1],pairs=item['pairs']))
display(pd.DataFrame(rows))
print('CPU PREPARATION COMPLETE. You may now switch to GPU.')
print(f'Saved workflow size: {sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file())/1e9:.3f} GB')


[Open stage 2: GPU paper analysis](https://colab.research.google.com/github/parmisbathayan/word-sentiment-eeg/blob/main/notebooks/zuco_paper_analyze_gpu.ipynb)